# Reproduce Main Results — IOSage

This notebook reproduces the key results from the paper:
1. Load pre-trained Phase 2 biquality model
2. Evaluate on held-out benchmark ground-truth test set (436 samples)
3. Compare against baselines (Drishti, Logistic Regression, majority class)
4. Generate the paper's main results table

**Expected runtime**: < 2 minutes (loads pre-trained models, no training)

In [1]:
import sys, pickle, numpy as np, pandas as pd
sys.path.insert(0, '..')
import yaml
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report
from sklearn.linear_model import LogisticRegression

DIMS = ['access_granularity', 'metadata_intensity', 'parallelism_efficiency',
        'access_pattern', 'interface_choice', 'file_strategy',
        'throughput_utilization', 'healthy']
DIM_SHORT = ['Granularity', 'Metadata', 'Parallelism', 'Pattern',
             'Interface', 'File Strat.', 'Throughput', 'Healthy']

## 1. Load Data and Models

In [2]:
# Load config
with open('../configs/training.yaml') as f:
    config = yaml.safe_load(f)

# Load production features for feature column names
prod_feat = pd.read_parquet('../data/processed/production/features.parquet')
prod_labels = pd.read_parquet('../data/processed/production/labels.parquet')
prod_labels = prod_labels.set_index('_jobid')
prod_feat = prod_feat.set_index('_jobid')
common = prod_feat.index.intersection(prod_labels.index)
prod_feat = prod_feat.loc[common]
prod_labels = prod_labels.loc[common]

exclude = set(config.get('exclude_features', []))
for col in prod_feat.columns:
    if col.startswith('_') or col.startswith('drishti_'):
        exclude.add(col)
feature_cols = [c for c in prod_feat.columns if c not in exclude]

# Load splits
with open('../data/processed/production/split_indices.pkl', 'rb') as f:
    splits = pickle.load(f)
train_idx = splits.get('train_idx', splits.get('train_indices'))

X_prod_train = prod_feat.iloc[train_idx][feature_cols].values.astype(np.float32)
y_prod_train = prod_labels.iloc[train_idx][DIMS].values.astype(np.float32)

# Load benchmark test set
test_feat = pd.read_parquet('../data/processed/benchmark/test_features.parquet')
test_labels = pd.read_parquet('../data/processed/benchmark/test_labels.parquet')

X_test = np.column_stack([
    test_feat[col].values if col in test_feat.columns else np.zeros(len(test_feat))
    for col in feature_cols
]).astype(np.float32)
y_test = test_labels[DIMS].values.astype(np.float32)

print(f'Production train: {len(X_prod_train)}')
print(f'Benchmark test: {len(X_test)}')
print(f'Features: {len(feature_cols)}')
print(f'Labels: {len(DIMS)}')

Production train: 91807
Benchmark test: 436
Features: 157
Labels: 8


## 2. Load Phase 2 Biquality Model and Evaluate

In [3]:
# Load best model
with open('../models/phase2/xgboost_biquality_w100.pkl', 'rb') as f:
    models = pickle.load(f)

# Predict
y_pred = np.zeros_like(y_test)
for i, dim in enumerate(DIMS):
    y_pred[:, i] = models[dim].predict(X_test)

micro = f1_score(y_test, y_pred, average='micro', zero_division=0)
macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
print(f'Phase 2 XGBoost (biquality):')
print(f'  Micro-F1: {micro:.4f}')
print(f'  Macro-F1: {macro:.4f}')
print()

# Per-label breakdown
print(f'{"Dimension":<28s} {"F1":>7s} {"Prec":>7s} {"Rec":>7s} {"Supp":>6s}')
print('-' * 55)
for i, dim in enumerate(DIMS):
    f1 = f1_score(y_test[:, i], y_pred[:, i], zero_division=0)
    p = precision_score(y_test[:, i], y_pred[:, i], zero_division=0)
    r = recall_score(y_test[:, i], y_pred[:, i], zero_division=0)
    s = int(y_test[:, i].sum())
    print(f'{dim:<28s} {f1:7.4f} {p:7.4f} {r:7.4f} {s:6d}')

Phase 2 XGBoost (biquality):
  Micro-F1: 0.9231
  Macro-F1: 0.8998

Dimension                         F1    Prec     Rec   Supp
-------------------------------------------------------
access_granularity            0.9260  0.9290  0.9231    156
metadata_intensity            1.0000  1.0000  1.0000     65
parallelism_efficiency        1.0000  1.0000  1.0000     50
access_pattern                0.6897  0.6250  0.7692     13
interface_choice              0.9559  0.9155  1.0000     65
file_strategy                 0.9032  1.0000  0.8235     34
throughput_utilization        0.8989  0.8889  0.9091     44
healthy                       0.8244  0.9474  0.7297     74


## 3. Baseline Comparison

In [4]:
from src.data.drishti_labeling import compute_drishti_codes, codes_to_labels

results = {}

# Majority class
y_maj = np.zeros_like(y_test)
for i in range(8):
    y_maj[:, i] = 1 if y_prod_train[:, i].mean() > 0.5 else 0
results['Majority Class'] = (f1_score(y_test, y_maj, average='micro', zero_division=0),
                              f1_score(y_test, y_maj, average='macro', zero_division=0))

# Logistic Regression
y_lr = np.zeros_like(y_test)
for i in range(8):
    lr = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
    lr.fit(X_prod_train, y_prod_train[:, i])
    y_lr[:, i] = lr.predict(X_test)
results['Logistic Regression'] = (f1_score(y_test, y_lr, average='micro', zero_division=0),
                                   f1_score(y_test, y_lr, average='macro', zero_division=0))

# Drishti heuristic
codes = compute_drishti_codes(test_feat)
drishti_labels = codes_to_labels(codes)
y_dr = drishti_labels[DIMS].values.astype(np.float32)
results['Drishti Heuristic'] = (f1_score(y_test, y_dr, average='micro', zero_division=0),
                                 f1_score(y_test, y_dr, average='macro', zero_division=0))

# Phase 1 (heuristic-only XGBoost)
with open('../models/xgboost_br_models.pkl', 'rb') as f:
    p1_models = pickle.load(f)
y_p1 = np.zeros_like(y_test)
for i, dim in enumerate(DIMS):
    y_p1[:, i] = p1_models[dim].predict(X_test)
results['Phase 1: XGBoost'] = (f1_score(y_test, y_p1, average='micro', zero_division=0),
                                f1_score(y_test, y_p1, average='macro', zero_division=0))

# Phase 2 (biquality)
results['Phase 2: Biquality'] = (micro, macro)

# Display
print(f'{"Method":<35s} {"Micro-F1":>10s} {"Macro-F1":>10s}')
print('-' * 55)
for name, (mi, ma) in results.items():
    print(f'{name:<35s} {mi:10.4f} {ma:10.4f}')

/projects/bdau/envs/sc2026/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/projects/bdau/envs/sc2026/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/projects/bdau/envs/sc2026/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/projects/bdau/envs/sc2026/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/projects/bdau/envs/sc2026/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/projects/bdau/envs/sc2026/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/projects/bdau/envs/sc2026/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


/projects/bdau/envs/sc2026/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Method                                Micro-F1   Macro-F1
-------------------------------------------------------
Majority Class                          0.1580     0.0363
Logistic Regression                     0.2935     0.2332
Drishti Heuristic                       0.3843     0.2827
Phase 1: XGBoost                        0.3850     0.2822
Phase 2: Biquality                      0.9231     0.8998


## 4. Verify Results Match Paper

Expected values from the paper:
- Phase 2 XGBoost: Micro-F1 = 0.923, Macro-F1 = 0.900
- Drishti baseline: Micro-F1 = 0.384
- Improvement: 2.4x over Drishti

In [5]:
assert abs(micro - 0.923) < 0.01, f'Micro-F1 mismatch: {micro:.4f} vs expected 0.923'
assert abs(macro - 0.900) < 0.01, f'Macro-F1 mismatch: {macro:.4f} vs expected 0.900'
print('All assertions passed — results match paper.')

All assertions passed — results match paper.
